In [33]:
#clarity of thought
#depth of analysis
#analysis

In [34]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field
from dotenv import load_dotenv
load_dotenv()


True

In [35]:
model=ChatOpenAI()


In [36]:
class output(BaseModel):
    score:float=Field(description="Score out of 10")
    summary:str=Field(description="Detailed analysis of the essay")
    

In [37]:
structured_model=model.with_structured_output(output)

d:\AgenticAI_Learning\LangGraph\venv\Lib\site-packages\langchain_openai\chat_models\base.py:2418: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [38]:
class UpscAns(TypedDict):
    essay:str
    clarity_of_thought: str
    clarity_of_thought_score: int
    depth_of_analysis_score: int
    language_score: int
    depth_of_analysis: str
    final_score: int
    final_feedback:str
    language:str



In [39]:
def cot(state:UpscAns)->UpscAns:
    prompt=f"Evaluate the clarity of thought in the following essay:\n{state['essay']}\nProvide a detailed analysis and use very less output tokens and also give a score out of 10"
    response=structured_model.invoke(prompt)
    state['clarity_of_thought']=response.summary
    state['clarity_of_thought_score']=response.score
    return {'clarity_of_thought':state['clarity_of_thought'],'clarity_of_thought_score':state['clarity_of_thought_score']}

In [40]:
def doa(state:UpscAns)->UpscAns:
    prompt=f"Evaluate the depth of analysis in the following essay:\n{state['essay']}\nProvide a detailed analysis and use very less output tokens and also give a score out of 10"
    response=structured_model.invoke(prompt)
    state['depth_of_analysis']=response.summary
    state['depth_of_analysis_score']=response.score
    return {'depth_of_analysis':state['depth_of_analysis'],'depth_of_analysis_score':state['depth_of_analysis_score']}

In [41]:
def lang(state:UpscAns)->UpscAns:
    prompt=f"Evaluate the language quality in the following essay:\n{state['essay']}\nProvide a detailed analysis and use very less output tokens and also give a score out of 10"
    response=structured_model.invoke(prompt)
    state['language']=response.summary
    state['language_score']=response.score
    return {'language':state['language'],'language_score':state['language_score']}

In [ ]:
def eval(state:UpscAns)->UpscAns:
    final_score=(state['clarity_of_thought_score']+state['depth_of_analysis_score']+state['language_score'])/3
    state['final_score']=final_score
    prompt=f"Provide a final feedback for the following essay based on the following analysis:\nClarity of Thought: {state['clarity_of_thought']} (Score: {state['clarity_of_thought_score']})\nDepth of Analysis: {state['depth_of_analysis']} (Score: {state['depth_of_analysis_score']})\nLanguage: {state['language']} (Score: {state['language_score']})\nFinal Score: {final_score}\nProvide a detailed feedback for the essay and use less tokens ."
    response=model.invoke(prompt)
    state['final_feedback']=response.summary
    return {'final_feedback':state['final_feedback'],'final_score':state['final_score']}

In [43]:
graph=StateGraph(UpscAns)

graph.add_node('COT',cot)
graph.add_node('DOA',doa)
graph.add_node('LANG',lang)
graph.add_node('EVAL',eval)

graph.add_edge(START,'COT')
graph.add_edge(START,'DOA')
graph.add_edge(START,'LANG')
graph.add_edge('COT','EVAL')
graph.add_edge('DOA','EVAL')
graph.add_edge('LANG','EVAL')
graph.add_edge('EVAL',END)

workflow=graph.compile()


In [44]:
initial_state={'essay':"The impact of climate change on global ecosystems is profound and multifaceted. Rising temperatures, changing precipitation patterns, and increased frequency of extreme weather events are altering habitats and threatening biodiversity worldwide. For instance, polar bears are losing their sea ice habitat, while coral reefs are experiencing bleaching due to warmer ocean temperatures. Additionally, shifts in plant and animal distributions are disrupting ecological interactions and food webs. Addressing climate change requires urgent action to reduce greenhouse gas emissions and implement adaptive strategies to protect vulnerable ecosystems."}

workflow.invoke(initial_state)


{'essay': 'The impact of climate change on global ecosystems is profound and multifaceted. Rising temperatures, changing precipitation patterns, and increased frequency of extreme weather events are altering habitats and threatening biodiversity worldwide. For instance, polar bears are losing their sea ice habitat, while coral reefs are experiencing bleaching due to warmer ocean temperatures. Additionally, shifts in plant and animal distributions are disrupting ecological interactions and food webs. Addressing climate change requires urgent action to reduce greenhouse gas emissions and implement adaptive strategies to protect vulnerable ecosystems.',
 'clarity_of_thought': 'The essay effectively communicates the profound and multifaceted impact of climate change on global ecosystems. The introduction clearly outlines the key points, and each subsequent paragraph elaborates on specific examples and consequences of climate change. The essay maintains a logical flow and transitions smooth